# SEC Browser Colab Workflow

Use this notebook to upload SEC `.asc` or `.xls` files, preview overlays inline, and export SVG, PNG, Plotly HTML, or XMGrace output without using the desktop browser.

## Workflow

1. Run the setup cell.
2. Upload your SEC files.
3. Review the detected file metadata.
4. Fill in the plotting parameters.
5. Run the export cell.
6. Download the generated files.

In [ ]:
!pip -q install matplotlib pandas plotly xlrd

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/elektra773/sec-browser.git"
REPO_DIR = Path("/content/sec-browser")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

UPLOAD_DIR = REPO_DIR / "colab_uploads"
OUTPUT_DIR = REPO_DIR / "colab_outputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo: {REPO_DIR}")
print(f"Uploads: {UPLOAD_DIR}")
print(f"Outputs: {OUTPUT_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from plot_sec_curves import (
    EXPORT_RASTER_DPI,
    PREVIEW_DPI,
    clean_label,
    export_plotly_html,
    export_xmgrace,
    extract_column_name,
    extract_run_date,
    find_top_peaks,
    process_trace,
    read_sec_file,
    render_sec_plot,
)

def rgb_to_hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(
        int(rgb[0] * 255), int(rgb[1] * 255), int(rgb[2] * 255)
    )

def list_sec_files(directory):
    return sorted(
        [p for p in directory.iterdir() if p.suffix.lower() in {".asc", ".xls"}],
        key=lambda p: p.name.lower(),
    )

In [ ]:
from google.colab import files

uploaded = files.upload()

for name, data in uploaded.items():
    (UPLOAD_DIR / name).write_bytes(data)

print(f"Saved {len(uploaded)} file(s) to {UPLOAD_DIR}")

In [ ]:
sec_files = list_sec_files(UPLOAD_DIR)

inventory = pd.DataFrame(
    [
        {
            "filename": path.name,
            "sample": clean_label(path),
            "column": extract_column_name(path),
            "date": extract_run_date(path),
            "type": path.suffix.lower().lstrip("."),
        }
        for path in sec_files
    ]
)

inventory

## Plot Parameters

Fill in `selected_files` before running the next cell. Use exact filenames from the inventory table above.

In [ ]:
selected_files = [
    # "example.asc",
]

labels = None
title = ""

normalize = True
normalize_anchor = "left-limit"  # "left-limit" or "zero"
baseline_subtract = True
smooth_window = 5

xlim = None          # Example: (5, 20)
ylim = None          # Example: (-0.05, 1.2)
figsize = (6, 4)
style = "paper"
transparent = False
show_legend = True

show_peak_ranks = set()   # Example: {1, 2, 3}

export_svg_png = True
export_plotly = True
export_xmgrace_file = False
output_stem = "sec_overlay"

In [ ]:
paths = [UPLOAD_DIR / name for name in selected_files]
assert paths, "Add at least one filename to selected_files first."

if labels is not None:
    assert len(labels) == len(paths), "labels must match selected_files length"

colors = plt.get_cmap("tab10").colors
processed_traces = []

for idx, path in enumerate(paths):
    raw_trace = read_sec_file(path)
    trace = process_trace(
        raw_trace,
        normalize=normalize,
        baseline_subtract=baseline_subtract,
        smooth_window=max(1, smooth_window),
        normalize_window=xlim if normalize else None,
        normalize_anchor=normalize_anchor,
    )
    color = rgb_to_hex(colors[idx % len(colors)])
    processed_traces.append(
        {
            "trace_id": str(path),
            "label": labels[idx] if labels else clean_label(path),
            "color": color,
            "line_width": 3.0,
            "data": trace,
            "peaks": find_top_peaks(trace, peak_window=xlim, top_n=5),
        }
    )

peak_visibility = {
    trace["trace_id"]: set(show_peak_ranks)
    for trace in processed_traces
}

fig, ax = plt.subplots(figsize=figsize, dpi=PREVIEW_DPI)
render_sec_plot(
    fig=fig,
    ax=ax,
    traces=processed_traces,
    title=title,
    normalized=normalize,
    xlim=xlim,
    ylim=ylim,
    show_legend=show_legend,
    style=style,
    visible_peak_ranks=peak_visibility,
)
plt.show()

output_base = OUTPUT_DIR / output_stem
generated_files = []

if export_svg_png:
    svg_output = output_base.with_suffix(".svg")
    png_output = output_base.with_suffix(".png")
    fig.savefig(svg_output, transparent=transparent)
    fig.savefig(png_output, dpi=EXPORT_RASTER_DPI, transparent=transparent)
    generated_files.extend([svg_output, png_output])

if export_plotly:
    html_output = output_base.with_suffix(".html")
    export_plotly_html(
        traces=processed_traces,
        output=html_output,
        title=title,
        x_label="Elution Volume (mL)",
        y_label="Normalized absorbance" if normalize else "Absorbance (mAU)",
        show_legend=show_legend and len(processed_traces) > 1,
        xlim=xlim,
        ylim=ylim,
    )
    generated_files.append(html_output)

if export_xmgrace_file:
    agr_output = output_base.with_suffix(".agr")
    export_xmgrace(
        traces=processed_traces,
        output=agr_output,
        title=title,
        x_label="Elution Volume (mL)",
        y_label="Normalized absorbance" if normalize else "Absorbance (mAU)",
        show_legend=show_legend and len(processed_traces) > 1,
        xlim=xlim,
        ylim=ylim,
    )
    generated_files.append(agr_output)

print("Generated files:")
for path in generated_files:
    print(path)

In [ ]:
from google.colab import files

for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)

# Uncomment to download everything one-by-one.
# for path in sorted(OUTPUT_DIR.iterdir()):
#     files.download(str(path))